# 01 — Preprocess Dataset
Samples 50,000 rows, builds TF-IDF + numeric features for Ridge/FNN and token
sequences for LSTM, then saves all artefacts to `data/`.

**Backend:** Keras v3 / PyTorch (set via KERAS_BACKEND env var)
**Split:** 70% train · 15% val · 15% test
> **Runtime: ~2 min**


In [1]:
# ── All imports ──────────────────────────────────────────────────────────
import os
os.environ["KERAS_BACKEND"] = "torch"

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import save_npz, hstack
import pickle, json


In [2]:
# ── Sample 50k rows ───────────────────────────────────────────────────────
SAMPLE_SIZE = 50_000
RANDOM_SEED = 42

print("Loading CSV...")
df = pd.read_csv("Hotel_Reviews.csv")
print(f"Full dataset: {df.shape[0]:,} rows")

df["days_since_review"] = (
    df["days_since_review"].astype(str).str.extract(r"(\d+)").astype(float))
df = df.dropna(subset=["lat","lng","Reviewer_Score"])
df["Negative_Review"] = df["Negative_Review"].replace("No Negative","")
df["Positive_Review"]  = df["Positive_Review"].replace("No Positive","")
df["combined_review"]  = (
    df["Positive_Review"].fillna("") + " " + df["Negative_Review"].fillna("")
).str.strip()

df_sample = df.sample(n=SAMPLE_SIZE, random_state=RANDOM_SEED).reset_index(drop=True)
print(f"Sample: {len(df_sample):,} rows")
print(df_sample["Reviewer_Score"].describe().round(3))


Loading CSV...
Full dataset: 515,738 rows
Sample: 50,000 rows
count    50000.000
mean         8.393
std          1.644
min          2.500
25%          7.500
50%          8.800
75%          9.600
max         10.000
Name: Reviewer_Score, dtype: float64


In [3]:
# ── Split 70/15/15 ───────────────────────────────────────────────────────
NUMERIC_FEATS = [
    "Review_Total_Positive_Word_Counts",
    "Review_Total_Negative_Word_Counts",
    "Average_Score",
    "Total_Number_of_Reviews_Reviewer_Has_Given",
    "days_since_review",
]

y = df_sample["Reviewer_Score"].values
X_tv, X_test, y_tv, y_test = train_test_split(
    df_sample, y, test_size=0.15, random_state=RANDOM_SEED)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.15/0.85, random_state=RANDOM_SEED)

print(f"Train:{len(y_train):,} | Val:{len(y_val):,} | Test:{len(y_test):,}")


Train:34,999 | Val:7,501 | Test:7,500


In [4]:
# ── TF-IDF features (for Ridge + FNN) ────────────────────────────────────
print("Fitting TF-IDF (5000 features, unigrams+bigrams)...")
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2),
                        min_df=3, sublinear_tf=True)
X_tr_tfidf = tfidf.fit_transform(X_train["combined_review"])
X_va_tfidf  = tfidf.transform(X_val["combined_review"])
X_te_tfidf  = tfidf.transform(X_test["combined_review"])

scaler = StandardScaler()
X_tr_num = scaler.fit_transform(X_train[NUMERIC_FEATS])
X_va_num  = scaler.transform(X_val[NUMERIC_FEATS])
X_te_num  = scaler.transform(X_test[NUMERIC_FEATS])

# Stack TF-IDF + numeric → dense float32 for FNN
X_tr_dense = hstack([X_tr_tfidf, X_tr_num]).toarray().astype("float32")
X_va_dense  = hstack([X_va_tfidf, X_va_num]).toarray().astype("float32")
X_te_dense  = hstack([X_te_tfidf, X_te_num]).toarray().astype("float32")

print(f"Feature matrix: {X_tr_dense.shape[0]:,} rows × {X_tr_dense.shape[1]:,} cols")


Fitting TF-IDF (5000 features, unigrams+bigrams)...
Feature matrix: 34,999 rows × 5,005 cols


In [5]:
# ── Token sequences for LSTM ──────────────────────────────────────────────
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_WORDS = 15_000   # vocabulary size
MAX_LEN   = 150      # sequence length

print("Tokenising sequences...")
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train["combined_review"])

seq_tr = pad_sequences(tokenizer.texts_to_sequences(X_train["combined_review"]),
                       maxlen=MAX_LEN, truncating="post", padding="post")
seq_va = pad_sequences(tokenizer.texts_to_sequences(X_val["combined_review"]),
                       maxlen=MAX_LEN, truncating="post", padding="post")
seq_te = pad_sequences(tokenizer.texts_to_sequences(X_test["combined_review"]),
                       maxlen=MAX_LEN, truncating="post", padding="post")

print(f"Sequences — train:{seq_tr.shape} val:{seq_va.shape} test:{seq_te.shape}")


Tokenising sequences...
Sequences — train:(34999, 150) val:(7501, 150) test:(7500, 150)


In [6]:
# ── Save all artefacts ────────────────────────────────────────────────────
os.makedirs("data", exist_ok=True)

# Dense feature matrices (FNN / Ridge)
np.save("data/X_train_dense.npy", X_tr_dense)
np.save("data/X_val_dense.npy",   X_va_dense)
np.save("data/X_test_dense.npy",  X_te_dense)

# Numeric-only arrays (LSTM side-input)
np.save("data/X_train_num.npy", X_tr_num.astype("float32"))
np.save("data/X_val_num.npy",   X_va_num.astype("float32"))
np.save("data/X_test_num.npy",  X_te_num.astype("float32"))

# Sequences (LSTM)
np.save("data/seq_train.npy", seq_tr)
np.save("data/seq_val.npy",   seq_va)
np.save("data/seq_test.npy",  seq_te)

# Targets
np.save("data/y_train.npy", y_train)
np.save("data/y_val.npy",   y_val)
np.save("data/y_test.npy",  y_test)

# Preprocessors
with open("data/tfidf.pkl",     "wb") as f: pickle.dump(tfidf,     f)
with open("data/scaler.pkl",    "wb") as f: pickle.dump(scaler,    f)
with open("data/tokenizer.pkl", "wb") as f: pickle.dump(tokenizer, f)

meta = {
    "SAMPLE_SIZE":   SAMPLE_SIZE,
    "MAX_WORDS":     MAX_WORDS,
    "MAX_LEN":       MAX_LEN,
    "NUMERIC_FEATS": NUMERIC_FEATS,
    "INPUT_DIM":     int(X_tr_dense.shape[1]),
    "train_size":    int(len(y_train)),
    "val_size":      int(len(y_val)),
    "test_size":     int(len(y_test)),
}
with open("data/meta.json","w") as f: json.dump(meta, f, indent=2)

print("\n✅ All artefacts saved to data/")
print(json.dumps(meta, indent=2))



✅ All artefacts saved to data/
{
  "SAMPLE_SIZE": 50000,
  "MAX_WORDS": 15000,
  "MAX_LEN": 150,
  "NUMERIC_FEATS": [
    "Review_Total_Positive_Word_Counts",
    "Review_Total_Negative_Word_Counts",
    "Average_Score",
    "Total_Number_of_Reviews_Reviewer_Has_Given",
    "days_since_review"
  ],
  "INPUT_DIM": 5005,
  "train_size": 34999,
  "val_size": 7501,
  "test_size": 7500
}
